# SAM2 promptable segmentation pilot

This Phase 4 pilot uses pretrained SAM 2.1 Hiera Tiny without training, fine-tuning, ground truth, or numerical accuracy metrics. Outputs are candidate masks, not ground truth.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src').is_dir() and (candidate / 'data' / 'raw' / 'first_sample').is_dir(): return candidate
    raise FileNotFoundError('Open this notebook from the thesis-project folder or its notebooks folder.')

ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT))
SAM2_RESULTS = ROOT / 'results' / 'sam2'


## Official implementation and device

The project uses Meta's official `facebookresearch/sam2` repository and its SAM 2.1 Tiny checkpoint. Tiny has 38.9M parameters and is selected for practicality on an M1 MacBook Air. Device selection prefers Apple MPS and falls back to CPU; CUDA is never assumed.

In [ ]:
import json, pandas as pd
metadata = json.loads((SAM2_RESULTS / 'run_metadata.json').read_text())
pd.DataFrame(metadata.items(), columns=['Setting', 'Value'])

## Prompt plan

The same five Phase 3 representative images are used. Each row records a single positive point, one bounding box, and two positive plus one negative point. The negative point targets nearby non-target vegetation or structure. Coordinates are documented so prompt interaction remains auditable.

In [ ]:
prompts = pd.read_csv(SAM2_RESULTS / 'prompts.csv')
selection = pd.read_csv(ROOT / 'results' / 'representative_images.csv').dropna(subset=['Filename'])
assert set(selection.Filename) == set(prompts.Filename)
prompts

## Run the pilot

The runner saves original, prompt visualization, predicted mask, and overlay for the single-point, box, and positive-plus-negative experiments. The SAM2 score is not a validated accuracy metric. Running this cell does not modify raw images.

In [ ]:
import subprocess
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'run_phase4_sam2.py')], check=True)

## Qualitative review boundary

Review target-plant isolation, leaf/stem/tendril preservation, background-vegetation and trellis rejection, blur response, and human interaction. Do not calculate IoU, Dice, precision, or recall without manually validated reference masks.

In [ ]:
pd.read_csv(SAM2_RESULTS / 'inference_times.csv')
pd.read_csv(SAM2_RESULTS / 'qualitative_summary.csv')

## Interpretation

Compare visually with Phase 3 only. A box or negative point can encode human knowledge about the intended plant, so better-looking results may reflect greater interaction rather than model autonomy. Record failure cases before deciding whether SAM2 is suitable for the next experiment.